# Module 16 — Time-Series Forecasting with SARIMAX: Transaction Volume Prediction

> **Track 2 · Revolut Fintech Specialization**  
> **Difficulty**: Intermediate → Advanced  
> **Runtime**: ~5 minutes  
> **Key Libraries**: Statsmodels, Pandas, NumPy, Plotly, Seaborn  
> **Prerequisite**: Module 3 (Time-Series Resampling)

## Table of Contents

1. [Business Context](#1-business-context)
2. [Setup & Imports](#2-setup--imports)
3. [Synthetic Transaction Volume Dataset](#3-synthetic-transaction-volume-dataset)
4. [Exploratory Time-Series Analysis](#4-exploratory-time-series-analysis)
5. [Seasonality Decomposition](#5-seasonality-decomposition)
6. [Stationarity Check (ADF Test)](#6-stationarity-check-adf-test)
7. [SARIMAX Model Training](#7-sarimax-model-training)
8. [Model Diagnostics](#8-model-diagnostics)
9. [Forecasting Future Transaction Volume](#9-forecasting-future-transaction-volume)
10. [Anomaly Detection on Forecast Residuals](#10-anomaly-detection-on-forecast-residuals)
11. [Visualization Dashboard](#11-visualization-dashboard)
12. [Key Business Takeaway](#12-key-business-takeaway)

---
## §1 · Business Context

### Why Forecast Transaction Volume at Revolut?

Accurate transaction volume forecasts drive critical business decisions:

| Use Case | Team | Impact |
|----------|------|--------|
| **Infrastructure scaling** | Engineering | Provision servers for peak transaction loads |
| **Fraud baseline** | Fraud Ops | Expected volume → flag unusual spikes/drops |
| **Revenue forecasting** | Finance | Transaction fees × volume = revenue |
| **Marketing ROI** | Growth | Measure campaign impact on transaction activity |
| **Anomaly detection** | Security | Forecast residuals flag system issues or fraud waves |

**SARIMAX** (Seasonal ARIMA with eXogenous regressors) is ideal because:
- Captures **weekly seasonality** (weekends vs. weekdays).
- Captures **yearly seasonality** (holidays, Black Friday).
- Handles **trend** (growing user base).
- Incorporates **external factors** (marketing campaigns, holidays).

In this notebook we will:
1. Generate 2 years of daily transaction volume with realistic seasonality.
2. Decompose the time series into trend, seasonal, and residual components.
3. Train a SARIMAX model with weekly and yearly seasonality.
4. Forecast 30 days into the future with confidence intervals.
5. Detect anomalies in forecast residuals (system issues, fraud waves).

---
## §2 · Setup & Imports

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import time
import warnings

np.random.seed(42)
warnings.filterwarnings("ignore")

# ── Statsmodels ──────────────────────────────────────────────────
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# ── Scikit-learn ─────────────────────────────────────────────────
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ── Visualization ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# ── Display settings ─────────────────────────────────────────────
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
sns.set_theme(style="whitegrid")

print("✓ All imports loaded")

---
## §3 · Synthetic Transaction Volume Dataset

We simulate **2 years (730 days) of daily transaction volume** with:
- **Upward trend** (growing user base).
- **Weekly seasonality** (lower on weekends).
- **Yearly seasonality** (holiday spikes, summer dips).
- **Noise** (random daily variation).

In [ ]:
# ── Configuration ────────────────────────────────────────────────
START_DATE = "2023-01-01"
END_DATE = "2024-12-31"
FORECAST_HORIZON = 30  # days to forecast

print(f"Generating transaction volume data:")
print(f"  Period: {START_DATE} → {END_DATE}")
print(f"  Frequency: Daily")

# Generate date range
dates = pd.date_range(start=START_DATE, end=END_DATE, freq="D")
n_days = len(dates)

print(f"  Total days: {n_days}")

# ── Base trend (upward, ~5% annual growth) ───────────────────────
base_volume = 100_000  # starting daily volume
trend = base_volume * (1 + 0.05 * np.arange(n_days) / 365)

# ── Weekly seasonality (lower on weekends) ───────────────────────
day_of_week = dates.dayofweek  # 0=Mon, 6=Sun
weekly_seasonal = np.where(day_of_week < 5, 1.0, 0.75)  # weekends -25%
weekly_effect = trend * (weekly_seasonal - 1)

# ── Yearly seasonality (holidays, summer) ────────────────────────
month = dates.month
yearly_seasonal = np.ones(n_days)
yearly_seasonal[month == 12] = 1.30  # December +30% (Christmas)
yearly_seasonal[month == 11] = 1.20  # November +20% (Black Friday)
yearly_seasonal[month.isin([7, 8])] = 0.85  # Summer -15%
yearly_effect = trend * (yearly_seasonal - 1)

# ── Noise (random daily variation) ───────────────────────────────
noise = np.random.normal(0, base_volume * 0.05, n_days)  # ±5% daily noise

# ── Combine ──────────────────────────────────────────────────────
volume = trend + weekly_effect + yearly_effect + noise
volume = np.maximum(volume, 0)  # no negative transactions

# ── Create DataFrame ─────────────────────────────────────────────
df = pd.DataFrame({
    "date": dates,
    "transaction_volume": np.round(volume).astype(int),
    "day_of_week": day_of_week,
    "month": month,
    "is_weekend": (day_of_week >= 5).astype(int),
})
df.set_index("date", inplace=True)

print(f"\n✓ Shape: {df.shape}")
print(f"  Mean daily volume: {df['transaction_volume'].mean():,.0f}")
print(f"  Std daily volume : {df['transaction_volume'].std():,.0f}")
print(f"  Min: {df['transaction_volume'].min():,.0f}")
print(f"  Max: {df['transaction_volume'].max():,.0f}")

df.head(10)

---
## §4 · Exploratory Time-Series Analysis

In [ ]:
# Plot full time series
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df.index, y=df["transaction_volume"],
    mode="lines",
    name="Daily Transaction Volume",
    line=dict(color="#636EFA", width=1.5),
))

# Add 7-day rolling average
rolling_avg = df["transaction_volume"].rolling(7).mean()
fig.add_trace(go.Scatter(
    x=df.index, y=rolling_avg,
    mode="lines",
    name="7-Day Rolling Average",
    line=dict(color="#EF553B", width=2.5),
))

fig.update_layout(
    title="Daily Transaction Volume (2 Years)",
    xaxis_title="Date",
    yaxis_title="Transaction Volume",
    height=450,
    legend=dict(orientation="h", y=1.12),
)
fig.show()

print("💡 Key patterns visible:")
print("   - Upward trend (growing user base)")
print("   - Weekly seasonality (dips on weekends)")
print("   - Yearly seasonality (December spikes, summer dips)")

In [ ]:
# Monthly aggregation
df_monthly = df.resample("M")["transaction_volume"].sum()

fig = px.bar(
    x=df_monthly.index, y=df_monthly.values,
    title="Monthly Transaction Volume",
    labels={"x": "Month", "y": "Transaction Volume"},
)
fig.update_layout(height=400)
fig.show()

print("💡 Monthly view shows clear seasonal patterns:")
print("   - Q4 (Oct-Dec) is the busiest (holidays)")
print("   - Q3 (Jul-Sep) is the slowest (summer)")

In [ ]:
# Weekly pattern
df_weekday = df.groupby("day_of_week")["transaction_volume"].mean()
days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

fig = px.bar(
    x=days, y=df_weekday.values,
    title="Average Transaction Volume by Day of Week",
    labels={"x": "Day of Week", "y": "Avg Daily Volume"},
    color=df_weekday.values,
    color_continuous_scale="Viridis",
)
fig.update_layout(height=400)
fig.show()

print("💡 Weekday pattern:")
print("   - Monday-Friday: high volume (business days)")
print("   - Saturday-Sunday: lower volume (weekends)")
print("   - Friday is often the busiest (payday effect)")

---
## §5 · Seasonality Decomposition

### STL Decomposition (Seasonal-Trend-Loess)

Decompose the time series into:
- **Trend**: long-term direction.
- **Seasonal**: repeating patterns (weekly + yearly).
- **Residual**: noise (what's left after removing trend and seasonality).

In [ ]:
# Perform seasonal decomposition
# Use period=7 for weekly seasonality
decomposition = seasonal_decompose(
    df["transaction_volume"],
    model="additive",  # additive model (trend + seasonal + residual)
    period=7,          # weekly seasonality
)

# Plot decomposition
fig = make_subplots(
    rows=4, cols=1,
    subplot_titles=["Observed", "Trend", "Seasonal (Weekly)", "Residual"],
    vertical_spacing=0.08,
)

fig.add_trace(go.Scatter(x=df.index, y=decomposition.observed, name="Observed",
                         line=dict(color="#636EFA")), row=1, col=1)
fig.add_trace(go.Scatter(x=df.index, y=decomposition.trend, name="Trend",
                         line=dict(color="#EF553B", width=2.5)), row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=decomposition.seasonal, name="Seasonal",
                         line=dict(color="#00CC96")), row=3, col=1)
fig.add_trace(go.Scatter(x=df.index, y=decomposition.resid, name="Residual",
                         line=dict(color="#FFA15A")), row=4, col=1)

fig.update_layout(height=700, showlegend=False)
fig.show()

print("💡 Decomposition reveals:")
print("   - Trend: steady upward growth")
print("   - Seasonal: weekly pattern (weekend dips)")
print("   - Residual: random noise (should be ~0 mean)")

---
## §6 · Stationarity Check (ADF Test)

### Augmented Dickey-Fuller Test

**Null hypothesis**: the time series has a unit root (non-stationary).
**p-value < 0.05**: reject null → stationary.

In [ ]:
def adf_test(series, name="Series"):
    """Perform ADF test and print results."""
    result = adfuller(series.dropna())
    
    print(f"{name}:")
    print(f"  ADF Statistic: {result[0]:.4f}")
    print(f"  p-value      : {result[1]:.6f}")
    print(f"  # Lags       : {result[2]}")
    print(f"  # Observations: {result[3]}")
    
    for key, value in result[4].items():
        print(f"  Critical Value ({key}): {value:.4f}")
    
    if result[1] < 0.05:
        print(f"  → Stationary (p < 0.05)")
    else:
        print(f"  → Non-stationary (p >= 0.05)")
    
    return result[1]

# Test original series
print("=" * 70)
print("STATIONARITY TEST (Augmented Dickey-Fuller)")
print("=" * 70)
p_original = adf_test(df["transaction_volume"], "Original Series")

# Test differenced series
print("\n" + "=" * 70)
df_diff = df["transaction_volume"].diff().dropna()
p_diff = adf_test(df_diff, "First Difference (ΔY)")

print("\n💡 SARIMAX handles non-stationarity via differencing (the 'I' in ARIMA)")
print("   We don't need to manually difference the data")

---
## §7 · SARIMAX Model Training

### Model Specification

**SARIMAX(p, d, q)(P, D, Q, s)** where:
- **(p, d, q)**: non-seasonal ARIMA parameters.
- **(P, D, Q, s)**: seasonal parameters (s = period).

For weekly seasonality (s=7):
- **(1, 1, 1)**: simple non-seasonal ARIMA.
- **(1, 1, 1, 7)**: seasonal ARIMA with weekly pattern.

In [ ]:
# Split into train/test
train_size = int(len(df) * 0.8)
df_train = df.iloc[:train_size]
df_test = df.iloc[train_size:]

print(f"Train: {df_train.index[0].date()} → {df_train.index[-1].date()} ({len(df_train)} days)")
print(f"Test : {df_test.index[0].date()} → {df_test.index[-1].date()} ({len(df_test)} days)")

# ── Define SARIMAX model ─────────────────────────────────────────
# Order: (p, d, q) for non-seasonal
# Seasonal order: (P, D, Q, s) for seasonal (s=7 for weekly)
order = (1, 1, 1)
seasonal_order = (1, 1, 1, 7)

print(f"\nSARIMAX Configuration:")
print(f"  Order         : {order}")
print(f"  Seasonal Order: {seasonal_order}")
print(f"  s = 7 (weekly seasonality)")

# ── Train model ──────────────────────────────────────────────────
print(f"\nTraining SARIMAX model …")
t0 = time.time()

model = SARIMAX(
    df_train["transaction_volume"],
    order=order,
    seasonal_order=seasonal_order,
    enforce_stationarity=False,
    enforce_invertibility=False,
)

results = model.fit(disp=False, maxiter=200)
train_time = time.time() - t0

print(f"✓ Trained in {train_time:.2f}s")
print(f"\nModel Summary:")
print(results.summary().tables[1])

---
## §8 · Model Diagnostics

### Check Residuals

Good model residuals should be:
1. **Zero mean** (no systematic bias).
2. **Normally distributed** (Gaussian noise).
3. **No autocorrelation** (independent over time).

In [ ]:
# Get residuals
residuals = results.resid

# Plot diagnostic plots
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=["Residuals Over Time", "Histogram", "Q-Q Plot", "ACF"],
)

# Residuals over time
fig.add_trace(go.Scatter(
    x=df_train.index, y=residuals,
    mode="lines", name="Residuals",
    line=dict(color="#636EFA", width=1),
), row=1, col=1)

# Histogram
fig.add_trace(go.Histogram(
    x=residuals, nbinsx=50, name="Residuals",
    marker_color="#EF553B",
), row=1, col=2)

# Q-Q plot (approximate)
from scipy import stats
(osm, osr), (slope, intercept, r) = stats.probplot(residuals)
fig.add_trace(go.Scatter(
    x=osm, y=osr, mode="markers", name="Q-Q",
    marker=dict(color="#00CC96", size=4),
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=[osm.min(), osm.max()],
    y=[slope * osm.min() + intercept, slope * osm.max() + intercept],
    mode="lines", name="Reference",
    line=dict(color="red", width=2),
), row=2, col=1)

# ACF
from statsmodels.tsa.stattools import acf
acf_values = acf(residuals, nlags=30)
fig.add_trace(go.Bar(
    x=list(range(len(acf_values))), y=acf_values,
    name="ACF", marker_color="#FFA15A",
), row=2, col=2)

fig.update_layout(height=600, showlegend=False)
fig.show()

# Residual statistics
print("Residual Statistics:")
print(f"  Mean: {residuals.mean():.2f}")
print(f"  Std : {residuals.std():.2f}")
print(f"  Skew: {residuals.skew():.4f}")
print(f"  Kurt: {residuals.kurtosis():.4f}")

print("\n💡 Good residuals:")
print("   - Mean ≈ 0 (no bias)")
print("   - Normally distributed (histogram)")
print("   - No autocorrelation (ACF bars within confidence bands)")

---
## §9 · Forecasting Future Transaction Volume

In [ ]:
# Forecast test period
forecast_test = results.get_forecast(steps=len(df_test))
forecast_test_mean = forecast_test.predicted_mean
forecast_test_ci = forecast_test.conf_int(alpha=0.05)  # 95% CI

# Evaluate on test set
mae = mean_absolute_error(df_test["transaction_volume"], forecast_test_mean)
rmse = np.sqrt(mean_squared_error(df_test["transaction_volume"], forecast_test_mean))
r2 = r2_score(df_test["transaction_volume"], forecast_test_mean)

print("=" * 70)
print("FORECAST EVALUATION (Test Set)")
print("=" * 70)
print(f"\nMAE : {mae:,.0f} transactions")
print(f"RMSE: {rmse:,.0f} transactions")
print(f"R²  : {r2:.4f}")

print(f"\n💡 Interpretation:")
print(f"   - MAE: average error is {mae:,.0f} transactions per day")
print(f"   - R²: model explains {r2*100:.1f}% of variance")

In [ ]:
# Forecast 30 days into the future
forecast_future = results.get_forecast(steps=FORECAST_HORIZON)
forecast_future_mean = forecast_future.predicted_mean
forecast_future_ci = forecast_future.conf_int(alpha=0.05)

print(f"\n30-Day Forecast Summary:")
print(f"  Total forecasted volume: {forecast_future_mean.sum():,.0f}")
print(f"  Average daily volume   : {forecast_future_mean.mean():,.0f}")
print(f"  Peak day               : {forecast_future_mean.max():,.0f}")
print(f"  Low day                : {forecast_future_mean.min():,.0f}")

# Show first 10 days
forecast_df = pd.DataFrame({
    "Date": forecast_future_mean.index,
    "Forecast": forecast_future_mean.values,
    "Lower CI (2.5%)": forecast_future_ci.iloc[:, 0].values,
    "Upper CI (97.5%)": forecast_future_ci.iloc[:, 1].values,
}).round(0)

print(f"\nFirst 10 Days of Forecast:")
print(forecast_df.head(10).to_string(index=False))

In [ ]:
# Visualize forecast
fig = go.Figure()

# Historical data
fig.add_trace(go.Scatter(
    x=df.index[-90:], y=df["transaction_volume"][-90:],
    mode="lines", name="Historical",
    line=dict(color="#636EFA", width=2),
))

# Forecast
fig.add_trace(go.Scatter(
    x=forecast_future_mean.index, y=forecast_future_mean,
    mode="lines", name="Forecast",
    line=dict(color="#EF553B", width=2.5),
))

# Confidence interval
fig.add_trace(go.Scatter(
    x=forecast_future_ci.index, y=forecast_future_ci.iloc[:, 1],
    mode="lines", name="Upper CI",
    line=dict(color="#EF553B", width=1, dash="dot"),
    showlegend=False,
))
fig.add_trace(go.Scatter(
    x=forecast_future_ci.index, y=forecast_future_ci.iloc[:, 0],
    mode="lines", name="Lower CI",
    line=dict(color="#EF553B", width=1, dash="dot"),
    fill="tonexty", fillcolor="rgba(239, 85, 59, 0.2)",
    showlegend=False,
))

fig.update_layout(
    title="30-Day Transaction Volume Forecast (with 95% Confidence Interval)",
    xaxis_title="Date",
    yaxis_title="Transaction Volume",
    height=500,
    legend=dict(orientation="h", y=1.12),
)
fig.show()

print("💡 The shaded area is the 95% confidence interval")
print("   Wider intervals = more uncertainty further into the future")

---
## §10 · Anomaly Detection on Forecast Residuals

### Logic

If actual volume falls **outside the 95% confidence interval**,
flag as an anomaly (system issue, fraud wave, marketing spike).

In [ ]:
# Compute residuals on full test set
test_residuals = df_test["transaction_volume"] - forecast_test_mean

# Flag anomalies (outside 95% CI)
lower_ci = forecast_test_ci.iloc[:, 0]
upper_ci = forecast_test_ci.iloc[:, 1]

anomalies = (df_test["transaction_volume"] < lower_ci) | (df_test["transaction_volume"] > upper_ci)

print(f"Anomaly Detection (Test Set):")
print(f"  Total days: {len(df_test)}")
print(f"  Anomalies flagged: {anomalies.sum()} ({anomalies.mean()*100:.2f}%)")
print(f"  Expected (95% CI): ~{0.05 * len(df_test):.0f} days")

# Show anomalies
if anomalies.sum() > 0:
    anomaly_dates = df_test[anomalies].index
    anomaly_volumes = df_test[anomalies]["transaction_volume"]
    forecast_values = forecast_test_mean[anomalies]
    
    print(f"\nAnomalous Days:")
    for i, (date, vol, fc) in enumerate(zip(anomaly_dates, anomaly_volumes, forecast_values)):
        direction = "↑" if vol > fc else "↓"
        print(f"  {date.date()} : {vol:>8,} (forecast: {fc:>8,.0f}) {direction}")
else:
    print("\n  No anomalies detected (all within 95% CI)")

In [ ]:
# Visualize anomalies
fig = go.Figure()

# Actual values
fig.add_trace(go.Scatter(
    x=df_test.index, y=df_test["transaction_volume"],
    mode="lines", name="Actual",
    line=dict(color="#636EFA", width=2),
))

# Forecast
fig.add_trace(go.Scatter(
    x=forecast_test_mean.index, y=forecast_test_mean,
    mode="lines", name="Forecast",
    line=dict(color="#EF553B", width=2.5),
))

# Confidence interval
fig.add_trace(go.Scatter(
    x=forecast_test_ci.index, y=forecast_test_ci.iloc[:, 1],
    mode="lines", name="Upper CI",
    line=dict(color="#EF553B", width=1, dash="dot"),
    showlegend=False,
))
fig.add_trace(go.Scatter(
    x=forecast_test_ci.index, y=forecast_test_ci.iloc[:, 0],
    mode="lines", name="Lower CI",
    line=dict(color="#EF553B", width=1, dash="dot"),
    fill="tonexty", fillcolor="rgba(239, 85, 59, 0.2)",
    showlegend=False,
))

# Anomalies
if anomalies.sum() > 0:
    fig.add_trace(go.Scatter(
        x=df_test[anomalies].index,
        y=df_test[anomalies]["transaction_volume"],
        mode="markers", name="Anomalies",
        marker=dict(color="red", size=10, symbol="star"),
    ))

fig.update_layout(
    title="Forecast vs. Actual (Anomalies Highlighted)",
    xaxis_title="Date",
    yaxis_title="Transaction Volume",
    height=500,
    legend=dict(orientation="h", y=1.12),
)
fig.show()

print("💡 Red stars = anomalies (actual volume outside 95% CI)")
print("   These could indicate:")
print("   - System issues (unexpected drop)")
print("   - Fraud waves (unexpected spike)")
print("   - Marketing campaigns (unexpected spike)")

---
## §11 · Visualization Dashboard

In [ ]:
# ── 11.1 Forecast Accuracy Over Time ─────────────────────────────
# Compute rolling MAE (7-day window)
rolling_mae = pd.Series(test_residuals).abs().rolling(7).mean()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_test.index, y=rolling_mae,
    mode="lines", name="7-Day Rolling MAE",
    line=dict(color="#636EFA", width=2.5),
))

fig.update_layout(
    title="Forecast Accuracy Over Time (7-Day Rolling MAE)",
    xaxis_title="Date",
    yaxis_title="Mean Absolute Error (transactions)",
    height=400,
)
fig.show()

print("💡 Rolling MAE shows forecast accuracy over time")
print("   Lower = better. Spikes = harder to predict periods")

In [ ]:
# ── 11.2 Business Impact Summary ─────────────────────────────────
avg_txn_value = 50  # £ average transaction value

total_forecast = forecast_future_mean.sum()
ci_lower = forecast_future_ci.iloc[:, 0].sum()
ci_upper = forecast_future_ci.iloc[:, 1].sum()

print("=" * 70)
print("BUSINESS IMPACT: 30-Day Forecast")
print("=" * 70)
print(f"\nAssumptions:")
print(f"  Average transaction value: £{avg_txn_value}")
print(f"  Transaction fee rate: 0.5%")

print(f"\n30-Day Forecast:")
print(f"  Transaction volume: {total_forecast:,.0f}")
print(f"  95% CI: [{ci_lower:,.0f}, {ci_upper:,.0f}]")

revenue_estimate = total_forecast * avg_txn_value * 0.005
revenue_ci_lower = ci_lower * avg_txn_value * 0.005
revenue_ci_upper = ci_upper * avg_txn_value * 0.005

print(f"\nEstimated Revenue (transaction fees):")
print(f"  Point estimate: £{revenue_estimate:,.0f}")
print(f"  95% CI: [£{revenue_ci_lower:,.0f}, £{revenue_ci_upper:,.0f}]")

print(f"\n💡 Use this forecast for:")
print(f"   - Infrastructure capacity planning")
print(f"   - Revenue forecasting")
print(f"   - Marketing budget allocation")

---
## §12 · Key Business Takeaway

> ### 🎯 Action Items for a Fintech Data Scientist
>
> | Aspect | Recommendation |
> |--------|---------------|
> | **When to use** | Daily/weekly transaction volume forecasting |
> | **Seasonality** | Always include weekly (s=7) and yearly (s=365) patterns |
> | **Model order** | Start with (1,1,1)(1,1,1,s), tune via AIC/BIC |
> | **Evaluation** | MAE, RMSE, R² on holdout test set |
> | **Anomaly detection** | Flag days outside 95% confidence interval |
>
> ### 💡 Revolut-Specific Guidelines
>
> 1. **Retrain weekly** on the latest 2 years of data:
>    - User behavior evolves (new features, market changes).
>    - Weekly retraining keeps the model current.
>
> 2. **Use forecast for infrastructure scaling**:
>    ```
>    If forecast > 150K transactions/day:
>        → Scale up server capacity
>    If forecast < 80K transactions/day:
>        → Scale down (cost savings)
>    ```
>
> 3. **Monitor forecast residuals for anomalies**:
>    - Actual volume outside 95% CI = investigate.
>    - Could indicate system issues, fraud waves, or marketing impact.
>
> 4. **Combine with exogenous variables** (SARIMAX):
>    - Marketing spend (TV, digital campaigns).
>    - Economic indicators (GDP, inflation).
>    - Competitor activity (new launches, promotions).
>
> ### 📌 SARIMAX Cheat Sheet
>
> ```python
> from statsmodels.tsa.statespace.sarimax import SARIMAX
>
> # Define model
> model = SARIMAX(
>     y_train,
>     order=(1, 1, 1),           # (p, d, q)
>     seasonal_order=(1, 1, 1, 7),  # (P, D, Q, s) for weekly
> )
>
> # Train
> results = model.fit(disp=False)
>
> # Forecast
> forecast = results.get_forecast(steps=30)
> forecast_mean = forecast.predicted_mean
> forecast_ci = forecast.conf_int(alpha=0.05)  # 95% CI
> ```
>
> ### 🔑 Key Takeaways
>
> 1. **SARIMAX captures trend + seasonality** in a single model.
> 2. **Weekly seasonality (s=7)** is critical for transaction data.
> 3. **95% confidence intervals** quantify forecast uncertainty.
> 4. **Anomaly detection** on forecast residuals flags system issues or fraud waves.
> 5. **Use forecasts for infrastructure scaling and revenue planning**.